# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Thallysson100/FlyRank_AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Plain-Language Rule Definition
To build a transparent baseline priority queue before training complex ML models, we combine **content staleness** with **historical engagement volume**. 

Based on our empirical signal audit, content experiences its sharpest increase in decay risk between **90 and 180 days** post-publication (peaking at a 12.18% drop rate), whereas older content ($360\text{d}+$) suffers from survival bias. Meanwhile, high prior engagement strongly predicts traffic drop risk ($\ge 60\%$ drop rate for pages with $\ge 500$ impressions).

A page is flagged for priority review if it meets two simultaneous conditions:
1. **Staleness Threshold:** Content age is $\ge 90$ days (`content_age_days >= 90`).
2. **Engagement Threshold:** Historical 30-day baseline search demand is $\ge 500$ impressions (`impressions_prev30 >= 500`).

To convert classification logic into a **ranked priority queue**, flagged pages are scored proportionally to their logarithmic search volume:

$$\text{baseline\_action\_score} = \mathbb{I}(\text{age} \ge 90) \times \mathbb{I}(\text{imp\_prev30} \ge 500) \times \log_{10}(1 + \text{imp\_prev30})$$

### Empirical Signal Verdicts
* **Signal 1: Content Age (`content_age_days`) — Verdict: MIXED**
  * *Reasoning:* Decline rate peaks between 90–180 days (12.18%) before dropping off for $360\text{d}+$ pages (3.48%) due to survival bias. Threshold adjusted to $\ge 90$ days to capture the true high-risk window.
* **Signal 2: Engagement Tier (`impressions_prev30`) — Verdict: CONFIRMED**
  * *Reasoning:* High-engagement tiers ($500\text{--}10,000+$ impressions) show consistently elevated decline rates ($60.09\%\text{--}63.55\%$) compared to low-engagement tiers ($46.75\%$).

### Reason Codes & Action Labels
* **`STALE_HIGH_DEMAND_DECAY`** (Action: `EDITORIAL_REFRESH_REVIEW`)
  * **Trigger:** `content_age_days >= 90` AND `impressions_prev30 >= 500`.
  * **Meaning:** Page has established search demand and has entered the empirical high-risk decay window.
* **`NO_ACTION_REQUIRED`** (Action: `MONITOR`)
  * **Trigger:** Page is fresh ($< 90$ days) or low engagement ($< 500$ impressions).

In [1]:
import os
import duckdb
import pandas as pd

# 1. Establish DuckDB connection to Hugging Face warehouse release
con = duckdb.connect()
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
dim_content = f"read_parquet('{REL}/dim_content.parquet')"
fact_query_90d = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

# 2. Audit Signal 1: Content Staleness vs Empirical Decline Rate
print("=== SIGNAL AUDIT 1: Content Age vs. Traffic Decline Rate ===")
query_signal1 = f"""
WITH content_age AS (
    SELECT 
        content_hash_id,
        DATEDIFF('day', content_created_date, '2026-06-30'::DATE) AS content_age_days,
        CASE 
            WHEN DATEDIFF('day', content_created_date, '2026-06-30'::DATE) <= 30 THEN '01_0-30d'
            WHEN DATEDIFF('day', content_created_date, '2026-06-30'::DATE) <= 90 THEN '02_31-90d'
            WHEN DATEDIFF('day', content_created_date, '2026-06-30'::DATE) <= 180 THEN '03_91-180d'
            WHEN DATEDIFF('day', content_created_date, '2026-06-30'::DATE) <= 360 THEN '04_181-360d'
            ELSE '05_360d+'
        END AS freshness_bucket
    FROM {dim_content}
),
query_perf AS (
    SELECT 
        content_hash_id,
        SUM(impressions_prev30) AS imp_prev30,
        SUM(impressions_last30) AS imp_last30
    FROM {fact_query_90d}
    GROUP BY content_hash_id
)
SELECT 
    ca.freshness_bucket,
    COUNT(*) AS total_pages,
    SUM(CASE WHEN qp.imp_prev30 >= 100 AND qp.imp_last30 < 0.8 * qp.imp_prev30 THEN 1 ELSE 0 END) AS n_declining,
    ROUND(AVG(CASE WHEN qp.imp_prev30 >= 100 AND qp.imp_last30 < 0.8 * qp.imp_prev30 THEN 1.0 ELSE 0.0 END) * 100, 2) AS decline_rate_pct
FROM content_age ca
LEFT JOIN query_perf qp ON ca.content_hash_id = qp.content_hash_id
GROUP BY ca.freshness_bucket
ORDER BY ca.freshness_bucket;
"""
df_signal1 = con.sql(query_signal1).df()
print(df_signal1.to_string(index=False))

# 3. Audit Signal 2: Historical Impression Volume vs Empirical Decline Rate
print("\n=== SIGNAL AUDIT 2: Prior Engagement Tier vs. Traffic Decline Rate ===")
query_signal2 = f"""
WITH volume_tiers AS (
    SELECT 
        content_hash_id,
        SUM(impressions_prev30) AS imp_prev30,
        SUM(impressions_last30) AS imp_last30,
        CASE 
            WHEN SUM(impressions_prev30) < 100 THEN '01_<100'
            WHEN SUM(impressions_prev30) <= 500 THEN '02_100-500'
            WHEN SUM(impressions_prev30) <= 2500 THEN '03_500-2500'
            WHEN SUM(impressions_prev30) <= 10000 THEN '04_2500-10000'
            ELSE '05_10000+'
        END AS volume_bucket
    FROM {fact_query_90d}
    GROUP BY content_hash_id
)
SELECT 
    vt.volume_bucket,
    COUNT(*) AS total_pages,
    SUM(CASE WHEN vt.imp_last30 < 0.8 * vt.imp_prev30 THEN 1 ELSE 0 END) AS n_declining,
    ROUND(AVG(CASE WHEN vt.imp_last30 < 0.8 * vt.imp_prev30 THEN 1.0 ELSE 0.0 END) * 100, 2) AS decline_rate_pct
FROM volume_tiers vt
GROUP BY vt.volume_bucket
ORDER BY vt.volume_bucket;
"""
df_signal2 = con.sql(query_signal2).df()
print(df_signal2.to_string(index=False))

=== SIGNAL AUDIT 1: Content Age vs. Traffic Decline Rate ===
freshness_bucket  total_pages  n_declining  decline_rate_pct
        01_0-30d        27935          0.0              0.00
       02_31-90d        58237       3230.0              5.55
      03_91-180d        78887       9607.0             12.18
     04_181-360d       187181      11876.0              6.34
        05_360d+       167366       5829.0              3.48

=== SIGNAL AUDIT 2: Prior Engagement Tier vs. Traffic Decline Rate ===
volume_bucket  total_pages  n_declining  decline_rate_pct
      01_<100        83765      39158.0             46.75
   02_100-500        29718      17857.0             60.09
  03_500-2500        14586       9047.0             62.03
04_2500-10000         4546       2889.0             63.55
    05_10000+         1237        749.0             60.55


## 2. Build the ranked queue (writes the CSV)

### Scoring & Ranking Logic
To construct our baseline priority queue, we compute the logarithmic engagement magnitude for all pages satisfying our empirical trigger rules (`content_age_days >= 90` and `impressions_prev30 >= 500`):

$$\text{baseline\_action\_score} = \begin{cases} \log_{10}(1 + \text{impressions\_prev30}) & \text{if } \text{content\_age\_days} \ge 90 \text{ AND } \text{impressions\_prev30} \ge 500 \\ 0.0 & \text{otherwise} \end{cases}$$

The priority queue sorts candidates primarily by `baseline_action_score` descending, with `impressions_prev30` descending as a secondary tie-breaker. Output is saved to `work/outputs/baseline_action_score.csv`.

In [2]:
import os
import duckdb
import pandas as pd

# 1. Root path resolution
export_path = "work/outputs/baseline_action_score.csv"

root = os.getcwd()
while not (
    os.path.isdir(os.path.join(root, "work"))
    or os.path.isdir(os.path.join(root, ".git"))
    or os.path.isdir(os.path.join(root, "skills"))
):
    parent = os.path.dirname(root)
    if parent == root:
        break
    root = parent

os.chdir(root)
os.makedirs("work/outputs", exist_ok=True)

# 2. Connect to DuckDB HF Warehouse
con = duckdb.connect()
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
dim_content = f"read_parquet('{REL}/dim_content.parquet')"
fact_query_90d = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

# 3. Build Ranked Priority Queue Query (Includes impressions_last30 in final SELECT)
queue_query = f"""
WITH content_base AS (
    SELECT 
        c.content_hash_id,
        c.client_hash_id,
        DATEDIFF('day', c.content_created_date, '2026-06-30'::DATE) AS content_age_days,
        COALESCE(q.imp_prev30, 0) AS impressions_prev30,
        COALESCE(q.imp_last30, 0) AS impressions_last30
    FROM {dim_content} c
    LEFT JOIN (
        SELECT 
            content_hash_id,
            SUM(impressions_prev30) AS imp_prev30,
            SUM(impressions_last30) AS imp_last30
        FROM {fact_query_90d}
        GROUP BY content_hash_id
    ) q ON c.content_hash_id = q.content_hash_id
)
SELECT 
    content_hash_id,
    client_hash_id,
    content_age_days,
    impressions_prev30,
    impressions_last30,
    CASE 
        WHEN content_age_days >= 90 AND impressions_prev30 >= 500 
        THEN ROUND(1.0 * LOG10(1 + impressions_prev30), 4)
        ELSE 0.0
    END AS baseline_action_score,
    CASE 
        WHEN content_age_days >= 90 AND impressions_prev30 >= 500 
        THEN 'STALE_HIGH_DEMAND_DECAY'
        ELSE 'NO_ACTION_REQUIRED'
    END AS reason_code,
    CASE 
        WHEN content_age_days >= 90 AND impressions_prev30 >= 500 
        THEN 'EDITORIAL_REFRESH_REVIEW'
        ELSE 'MONITOR'
    END AS action_label,
    -- Validation Target: True decline status (imp_last30 < 0.8 * imp_prev30)
    CASE 
        WHEN impressions_prev30 >= 300 AND impressions_last30 < 0.8 * impressions_prev30 THEN 1 
        ELSE 0 
    END AS is_declining_label
FROM content_base
ORDER BY baseline_action_score DESC, impressions_prev30 DESC;
"""

# 4. Execute query and write to CSV
queue_df = con.sql(queue_query).df()
queue_df.to_csv(export_path, index=False)

# 5. Output Summary Metrics
print(f"Total Pages Processed: {len(queue_df):,}")
print(f"Total Actionable Items Flagged (EDITORIAL_REFRESH_REVIEW): {(queue_df['action_label'] == 'EDITORIAL_REFRESH_REVIEW').sum():,}")
print(f"Ranked Priority Queue written to: {export_path}\n")

print("Top 5 Items in Priority Queue:")
print(queue_df[['content_hash_id', 'content_age_days', 'impressions_prev30', 'impressions_last30', 'baseline_action_score', 'reason_code', 'action_label']].head(5).to_string(index=False))

Total Pages Processed: 519,606
Total Actionable Items Flagged (EDITORIAL_REFRESH_REVIEW): 18,230
Ranked Priority Queue written to: work/outputs/baseline_action_score.csv

Top 5 Items in Priority Queue:
         content_hash_id  content_age_days  impressions_prev30  impressions_last30  baseline_action_score             reason_code             action_label
content_65c75874a23fca87               174            185602.0             20756.0                 5.2686 STALE_HIGH_DEMAND_DECAY EDITORIAL_REFRESH_REVIEW
content_39e19a3ec2d95f9d               160            177963.0              6622.0                 5.2503 STALE_HIGH_DEMAND_DECAY EDITORIAL_REFRESH_REVIEW
content_c60628276389acbb               119            160312.0            122585.0                 5.2050 STALE_HIGH_DEMAND_DECAY EDITORIAL_REFRESH_REVIEW
content_15168cd1c94e3529               139            133285.0             41887.0                 5.1248 STALE_HIGH_DEMAND_DECAY EDITORIAL_REFRESH_REVIEW
content_987d251ee617d9c

## 3. Top-20 review

### Priority Queue Top-20 Inspection
The table below details the top 20 ranked pages extracted directly from `baseline_action_score.csv`. Each entry presents the page hash, content age, historical search demand (`impressions_prev30`), baseline score, assigned action label, reason code, actual trend trajectory, and potential failure modes.

* **Action:** `EDITORIAL_REFRESH_REVIEW` (All top 20 pages exceed the 90-day age and 500-impression thresholds).
* **Reason Code:** `STALE_HIGH_DEMAND_DECAY`
* **Confidence Note:** High confidence for content optimization ROI due to significant established search demand ($\ge 500$ impressions).

In [3]:
import pandas as pd

# 1. Read exported queue CSV
df_queue = pd.read_csv('work/outputs/baseline_action_score.csv')

# 2. Extract Top 20 rows
top_20 = df_queue.head(20).copy()

# 3. Calculate % impression change between windows
top_20['imp_change_pct'] = (
    (top_20['impressions_last30'] - top_20['impressions_prev30']) 
    / top_20['impressions_prev30'].replace(0, 1) * 100
).round(2)

# 4. Display formatted table
display_cols = [
    'content_hash_id', 
    'content_age_days', 
    'impressions_prev30', 
    'impressions_last30', 
    'imp_change_pct',
    'baseline_action_score', 
    'reason_code', 
    'action_label',
    'is_declining_label'
]

print("=== TOP 20 PRIORITY QUEUE ITEMS FOR REVIEW ===")
print(top_20[display_cols].to_string(index=False))

=== TOP 20 PRIORITY QUEUE ITEMS FOR REVIEW ===
         content_hash_id  content_age_days  impressions_prev30  impressions_last30  imp_change_pct  baseline_action_score             reason_code             action_label  is_declining_label
content_65c75874a23fca87               174            185602.0             20756.0          -88.82                 5.2686 STALE_HIGH_DEMAND_DECAY EDITORIAL_REFRESH_REVIEW                   1
content_39e19a3ec2d95f9d               160            177963.0              6622.0          -96.28                 5.2503 STALE_HIGH_DEMAND_DECAY EDITORIAL_REFRESH_REVIEW                   1
content_c60628276389acbb               119            160312.0            122585.0          -23.53                 5.2050 STALE_HIGH_DEMAND_DECAY EDITORIAL_REFRESH_REVIEW                   1
content_15168cd1c94e3529               139            133285.0             41887.0          -68.57                 5.1248 STALE_HIGH_DEMAND_DECAY EDITORIAL_REFRESH_REVIEW                   

### Priority Queue Top-20 Line-by-Line Inspection

All top 20 pages carry the action **`EDITORIAL_REFRESH_REVIEW`** and reason code **`STALE_HIGH_DEMAND_DECAY`** because every item satisfies `content_age_days >= 90` and `impressions_prev30 >= 500`. The table below provides a qualitative line-by-line review of each item in the queue, evaluating confidence, observed trend trajectories, and primary failure modes (*what would make the flag wrong*).

| Rank | Content Hash ID | Age (d) | Prev 30d Imp | Change (%) | Trend Label | Confidence Note & What Would Make It Wrong (Failure Modes) |
| --- | --- | --- | --- | --- | --- | --- |
| **1** | `content_65c75874a23fca87` | 174 | 185,602 | -88.82% | Declining | **High Confidence.** Massive impression crash (-88.8%). *Wrong if:* Caused by technical indexing loss, broken tracking, or URL canonicalization errors rather than stale copy.|
| **2** | `content_39e19a3ec2d95f9d` | 160 | 177,963 | -96.28% | Declining | **High Confidence.** Traffic almost entirely collapsed (-96.3%). *Wrong if:* Page lost a featured snippet or SERP layout element that copy updates alone cannot recover.|
| **3** | `content_c60628276389acbb` | 119 | 160,312 | -23.53% | Declining | **Medium Confidence.** Meets decline threshold ($> 20\%$), but page still retains 122k impressions. *Wrong if:* Drop represents normal macro-search fluctuation; refreshing risks destabilizing evergreen traffic.|
| **4** | `content_15168cd1c94e3529` | 139 | 133,285 | -68.57% | Declining | **High Confidence.** Sharp impression loss (-68.6%). *Wrong if:* User search intent fundamentally shifted to non-text formats (e.g., video/tools) rather than outdated text.|
| **5** | `content_987d251ee617d9c6` | 145 | 131,885 | -43.35% | Declining | **High Confidence.** Consistent erosion (-43.4%). *Wrong if:* Drop stems from off-page backlink decay or competitor domain authority gain rather than on-page content quality.|
| **6** | `content_44f34c0a90047651` | 160 | 131,836 | -81.03% | Declining | **High Confidence.** Severe drop (-81.0%). *Wrong if:* Seasonal interest peak ended during `impressions_prev30` and baseline represents an artificial high.|
| **7** | `content_9648c4d1595a0794` | 301 | 118,496 | -99.50% | Declining | **High Confidence.** Near-total traffic wipeout (118k down to 595). *Wrong if:* Page was unpublished, redirected, or hit by an explicit manual penalty.|
| **8** | `content_4c15fe8dc370f3cd` | 308 | 112,979 | -42.46% | Declining | **High Confidence.** Moderate decay (-42.5%) on older content. *Wrong if:* Information remains factually current and traffic loss is purely keyword cannibalization from new internal posts.|
| **9** | `content_33d31496fca9665e` | 278 | 112,574 | -14.86% | **Stable (FP)** | **Low Confidence (False Positive).** Traffic down only 14.9% (remains at 95k impressions). *Wrong if:* Flagged solely due to raw volume; page is stable evergreen content.|
| **10** | `content_21309e9a83c83653` | 466 | 111,575 | -37.30% | Declining | **High Confidence.** Steady decline over 466 days (-37.3%). *Wrong if:* Page topic is a historical event/archival record that does not require editorial updating.|
| **11** | `content_bddfdd871aa09fbe` | 169 | 110,973 | -57.82% | Declining | **High Confidence.** Substantial decline (-57.8%). *Wrong if:* Traffic drop reflects brand keyword volume changes outside site control.|
| **12** | `content_a9322b74ca7cb1bb` | 116 | 108,461 | -74.92% | Declining | **High Confidence.** Heavy decay (-74.9%). *Wrong if:* Product/offer discussed on page went out of stock, causing transient user drop.|
| **13** | `content_c1f764a2f362d1c3` | 116 | 101,220 | +113.55% | **Growing (FP)** | **Zero Confidence (False Positive).** Impressions doubled (101k to 216k). *Wrong if:* Flagged purely due to baseline volume rule; editing this page risks breaking viral momentum.|
| **14** | `content_f33ad8a343180e8b` | 174 | 96,323 | -43.49% | Declining | **High Confidence.** Solid decay candidate (-43.5%). *Wrong if:* Search engine core update reshuffled rankings sitewide regardless of page age.|
| **15** | `content_16b29599a771ed7d` | 232 | 90,462 | -3.57% | **Stable (FP)** | **Low Confidence (False Positive).** Highly stable page (-3.6%). *Wrong if:* Waste of editorial resources to re-evaluate healthy core content.|
| **16** | `content_f107e54b10b43725` | 176 | 90,066 | -2.59% | **Stable (FP)** | **Low Confidence (False Positive).** Fully intact demand (-2.6%). *Wrong if:* Volume rule misinterprets total scale as decay probability.|
| **17** | `content_661a7734f691bef5` | 174 | 89,435 | -39.84% | Declining | **High Confidence.** Clear traffic erosion (-39.8%). *Wrong if:* Primary keywords experienced macro-category demand decline.|
| **18** | `content_eadb33b5df496f4a` | 466 | 88,175 | -3.34% | **Stable (FP)** | **Low Confidence (False Positive).** Extremely consistent traffic (-3.3%) across 466 days. *Wrong if:* Volume bias flags top performing pillar pages.|
| **19** | `content_11bf4c33adea7bdc` | 111 | 87,461 | +21.63% | **Growing (FP)** | **Zero Confidence (False Positive).** Page gained +21.6% traffic. *Wrong if:* Baseline rule forces review on accelerating content.|
| **20** | `content_471d9cabce329a66` | 466 | 85,885 | -18.23% | **Stable (FP)** | **Low Confidence (False Positive).** Below the 20% decline threshold (-18.2%). *Wrong if:* Minor fluctuation is mistaken for genuine content decay.|

---

1. **Volume Bias in the Naive Baseline:** Out of the top 20 items prioritized by volume, **7 items (35%) are false positives** (either growing or fluctuating within $<20\%$ drop). The baseline rule successfully identifies high-impact pages, but confuses overall traffic scale with decay likelihood.


2. **Actionability Thresholds:** Highly decaying pages (e.g., Rank 1, 2, 7 with $>88\%$ traffic losses) represent high-priority editorial candidates, provided technical issues or seasonal keyword shifts are ruled out first.

## 4. Weak picks + leakage check

### Weak Picks Assessment
Our baseline is extremely naive: it assumes **all** popular, older content is decaying. Because we rank the priority queue strictly by baseline demand (`impressions_prev30`), our highest-scored items include highly successful **evergreen content** that hasn't actually crashed. 
* *Example from Top 5:* `content_c60628276389acbb` is flagged for urgent review because it had 160k prior impressions. However, it still maintained 122k impressions in the last 30 days. Flagging stable evergreen pages is a false positive that wastes editorial resources.

### Target Leakage Check
We can confirm **no future windows leaked** into the decision logic. 
* The priority queue sorting metric (`baseline_action_score`) and the flag (`action_label`) were calculated strictly using `content_age_days` and `impressions_prev30`. 
* The future target window (`impressions_last30`) was completely isolated from the scoring function and only used to generate the validation column (`is_declining_label`).

In [4]:
import pandas as pd

# 1. Load the queue
df_queue = pd.read_csv('work/outputs/baseline_action_score.csv')

# 2. Quantify "Weak Picks" (False Positives)
# Pages we flagged for review, but which did NOT actually suffer a >= 20% traffic drop
flagged_mask = df_queue['action_label'] == 'EDITORIAL_REFRESH_REVIEW'
false_positives = df_queue[flagged_mask & (df_queue['is_declining_label'] == 0)]

print("=== WEAK PICKS (FALSE POSITIVES) ===")
print(f"Total Items Flagged: {flagged_mask.sum():,}")
print(f"Items Flagged that DID NOT Decline: {len(false_positives):,}")
print(f"Baseline False Positive Rate: {len(false_positives) / flagged_mask.sum() * 100:.2f}%\n")

# 3. Leakage Check (Verify scoring columns)
print("=== LEAKAGE VERIFICATION ===")
sample_scores = df_queue[['baseline_action_score', 'impressions_prev30', 'impressions_last30']].head(5)
print("Notice that high baseline_action_score correlates ONLY with impressions_prev30, not last30:")
print(sample_scores.to_string(index=False))

=== WEAK PICKS (FALSE POSITIVES) ===
Total Items Flagged: 18,230
Items Flagged that DID NOT Decline: 6,567
Baseline False Positive Rate: 36.02%

=== LEAKAGE VERIFICATION ===
Notice that high baseline_action_score correlates ONLY with impressions_prev30, not last30:
 baseline_action_score  impressions_prev30  impressions_last30
                5.2686            185602.0             20756.0
                5.2503            177963.0              6622.0
                5.2050            160312.0            122585.0
                5.1248            133285.0             41887.0
                5.1202            131885.0             74712.0


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.